# E0-T04 — locked Colab GPU smoke test
Run top-to-bottom in a **fresh GPU runtime**. This notebook installs the committed lock, captures the runtime, loads only one model at a time, and writes non-sensitive pass/fail manifests to Drive. A Hugging Face token is optional because the pinned models are public.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Before continuing, add a My Drive shortcut to the shared `CMPE253-Project` folder if it is not already visible. Do not paste tokens or private paths into committed files.

In [ ]:
import os
from pathlib import Path

REPO = Path("/content/cmpe253-llm-demographic-bias-audit")
shared_root = os.environ.get("AUDIT_SHARED_ROOT") or input(
    "Mounted CMPE253 shared-folder path: "
).strip()
SHARED = Path(shared_root).expanduser().resolve()
ACTOR = "vineet"  # Abhinandan must change this to "abhinandan"
assert ACTOR in {"vineet", "abhinandan"}
assert SHARED.is_dir(), (
    "Add the shared-folder shortcut to My Drive or update SHARED for this runtime only"
)

In [ ]:
!if [ -d "{REPO}/.git" ]; then git -C "{REPO}" pull --ff-only; else git clone https://github.com/SJSUF25/cmpe253-llm-demographic-bias-audit.git "{REPO}"; fi  # noqa: E501
%cd {REPO}
!git rev-parse HEAD

In [ ]:
!pip -q install uv
!uv sync --frozen --group dev --group colab

In [ ]:
!nvidia-smi
!PYTHONPATH=src uv run --frozen --group colab python -m environment_tools capture --output "{SHARED}/manifests/e0_t04_{ACTOR}_environment.json"  # noqa: E501

## M1: Qwen
This process exits after the run, so its model memory is released before M2 starts. The test performs two passes over 20 prompts plus repeated likelihood scoring and fails on any tolerance violation.

In [ ]:
!PYTHONPATH=src uv run --frozen --group colab python -m inference.smoke_test --model-key M1 --output "{SHARED}/manifests/e0_t04_{ACTOR}_M1_smoke.json"  # noqa: E501

## M2: Phi

In [ ]:
!PYTHONPATH=src uv run --frozen --group colab python -m inference.smoke_test --model-key M2 --output "{SHARED}/manifests/e0_t04_{ACTOR}_M2_smoke.json"  # noqa: E501

In [ ]:
import json

for key in ("M1", "M2"):
    path = SHARED / "manifests" / f"e0_t04_{ACTOR}_{key}_smoke.json"
    result = json.loads(path.read_text())
    assert result["status"] == "pass", result
    print(key, result["status"], result["fixture"])